In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/telco_clean.csv").set_index("CustomerID")
target = "Churn Value"
y = df[target]
df.shape

(7043, 28)

In [2]:
addons = ["Online Security", "Online Backup", "Device Protection",
          "Tech Support", "Streaming TV", "Streaming Movies"]

print(((df["Phone Service"] == "No") == (df["Multiple Lines"] == "No phone service")).all())
print(all(((df[c] == "No internet service") == (df["Internet Service"] == "No")).all() for c in addons))

True
True


In [4]:
feat = pd.DataFrame(index=df.index)   # an empty table with the same customers

yes_no_cols = ["Senior Citizen", "Partner", "Dependents", "Paperless Billing"] + addons
for c in yes_no_cols:
    feat[c] = (df[c] == "Yes").astype(int)

feat["Tenure Months"] = df["Tenure Months"]
feat["Monthly Charges"] = df["Monthly Charges"]
feat["Total Charges"] = df["Total Charges"]
feat.shape

(7043, 13)

In [5]:
# Tenure bucket: bends the "early cliff" into something simple models can see
bins = [-1, 6, 12, 24, 48, 72]
labels = ["0-6", "7-12", "13-24", "25-48", "49-72"]
feat["Tenure Bucket"] = pd.cut(df["Tenure Months"], bins=bins, labels=labels)

# Protection count: how many of the 4 protection add-ons the customer has
protect = ["Online Security", "Online Backup", "Device Protection", "Tech Support"]
feat["Protection Count"] = feat[protect].sum(axis=1)

print(feat["Tenure Bucket"].value_counts().sort_index())
print(feat["Protection Count"].value_counts().sort_index())

Tenure Bucket
0-6      1481
7-12      705
13-24    1024
25-48    1594
49-72    2239
Name: count, dtype: int64
Protection Count
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64


In [6]:
# Tenure bucket: bends the "early cliff" into something simple models can see
bins = [-1, 6, 12, 24, 48, 72]
labels = ["0-6", "7-12", "13-24", "25-48", "49-72"]
feat["Tenure Bucket"] = pd.cut(df["Tenure Months"], bins=bins, labels=labels)

# Protection count: how many of the 4 protection add-ons the customer has
protect = ["Online Security", "Online Backup", "Device Protection", "Tech Support"]
feat["Protection Count"] = feat[protect].sum(axis=1)

print(feat["Tenure Bucket"].value_counts().sort_index())
print(feat["Protection Count"].value_counts().sort_index())

Tenure Bucket
0-6      1481
7-12      705
13-24    1024
25-48    1594
49-72    2239
Name: count, dtype: int64
Protection Count
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64


In [7]:
has_history = df["Tenure Months"] > 0

avg_revenue = (df["Total Charges"] / df["Tenure Months"]).where(has_history, df["Monthly Charges"])
charge_change = df["Monthly Charges"] - avg_revenue
ratio = (df["Monthly Charges"] / df["Total Charges"]).where(df["Total Charges"] > 0, 1.0)

candidates = pd.DataFrame({
    "Avg Monthly Revenue": avg_revenue,
    "Charge Change": charge_change,
    "Abs Charge Change": charge_change.abs(),
    "Monthly to Total Ratio": ratio,
})

print("Missing:", candidates.isna().sum().sum())
print("Avg revenue vs Monthly Charges:", round(candidates["Avg Monthly Revenue"].corr(df["Monthly Charges"]), 3))
print("Ratio vs 1/tenure:", round(ratio[has_history].corr(1 / df.loc[has_history, "Tenure Months"]), 3))
print(candidates.corrwith(y).round(3))

Missing: 0
Avg revenue vs Monthly Charges: 0.996
Ratio vs 1/tenure: 0.999
Avg Monthly Revenue       0.193
Charge Change             0.002
Abs Charge Change         0.127
Monthly to Total Ratio    0.313
dtype: float64


In [8]:
big = candidates["Abs Charge Change"] > 5
long_term = df["Tenure Months"] > 12
print(round(y[big & long_term].mean(), 3), round(y[~big & long_term].mean(), 3), (big & long_term).sum())

0.392 0.167 102


In [9]:
feat["Abs Charge Change"] = candidates["Abs Charge Change"]

In [10]:
X = pd.get_dummies(feat, columns=["Tenure Bucket"], drop_first=True, dtype=int)

for c in ["Multiple Lines", "Internet Service", "Contract", "Payment Method"]:
    X = X.join(pd.get_dummies(df[c], prefix=c, drop_first=True, dtype=int))

print(X.shape)
list(X.columns)

(7043, 28)


['Senior Citizen',
 'Partner',
 'Dependents',
 'Paperless Billing',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Tenure Months',
 'Monthly Charges',
 'Total Charges',
 'Protection Count',
 'Abs Charge Change',
 'Tenure Bucket_7-12',
 'Tenure Bucket_13-24',
 'Tenure Bucket_25-48',
 'Tenure Bucket_49-72',
 'Multiple Lines_No phone service',
 'Multiple Lines_Yes',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Contract_One year',
 'Contract_Two year',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check']

In [11]:
print("Missing values:", X.isna().sum().sum())
print("Non-numeric columns:", X.select_dtypes(exclude="number").shape[1])

out = X.join(y)
out.to_csv("../data/processed/telco_features.csv")

check = pd.read_csv("../data/processed/telco_features.csv", index_col="CustomerID")
print(check.shape, round(check[target].mean(), 4))

Missing values: 0
Non-numeric columns: 0
(7043, 29) 0.2654


In [12]:
check.corr()[target].drop(target).sort_values(key=abs, ascending=False).head(8).round(3)

Tenure Months                     -0.352
Internet Service_Fiber optic       0.308
Contract_Two year                 -0.302
Payment Method_Electronic check    0.302
Tenure Bucket_49-72               -0.263
Dependents                        -0.249
Internet Service_No               -0.228
Total Charges                     -0.198
Name: Churn Value, dtype: float64